# Notebook 00 — Cluster Setup: Deploy MLflow (Minimal Mode)

**RHOAI 3.5 | Admin / cluster-setup notebook**

> Run this notebook **once per namespace** before any of the other notebooks.
> It requires `cluster-admin` or `oc` access with permissions to edit the `DataScienceCluster` CR.

This notebook walks through:
1. Enabling the MLflow Operator in RHOAI
2. Deploying a **Minimal** `MLflowTrackingServer` (SQLite + PVC) — the fastest path to a working server for dev/test
3. Verifying the deployment and extracting the tracking URL
4. Showing the **Production** configuration (PostgreSQL + S3) for reference
5. Configuring per-project S3 artifact storage via `MLflowConfig`

---

## Deployment Modes at a Glance

| | Minimal | Production |
|---|---------|------------|
| **Database** | SQLite on a PVC | External PostgreSQL |
| **Artifact storage** | PVC (local) | S3-compatible object store |
| **Replicas** | 1 | 2+ |
| **serveArtifacts** | Not required | `true` (proxy to S3) |
| **Use case** | Dev, test, demo | Production workloads |
| **Data persistence** | PVC lifecycle | Managed DB + S3 |
| **HA / failover** | No | Yes |

> **Important:** The Minimal mode stores all data on a PVC attached to a single pod.
> Deleting the PVC or the namespace will destroy all experiment data.

## Prerequisites

- `oc` CLI installed and logged in: `oc login https://<api-url> -u <user> -p <password>`
- Cluster-admin or equivalent permissions on the target namespace
- RHOAI 3.5 installed

All commands in this notebook are shown as shell cells using the `!` prefix (Jupyter shell escape).
They call `oc` directly. Alternatively, copy the YAML blocks and run them in a terminal.

In [ ]:
# Verify oc is available and you are logged in
!oc whoami
!oc version --client

In [ ]:
# Set the target namespace — change this to your OpenShift project name
NAMESPACE = "my-mlflow-project"  # <--- EDIT THIS

import subprocess

def oc(cmd: str, capture: bool = False) -> str:
    """Run an oc command and optionally return its output."""
    result = subprocess.run(
        ["oc"] + cmd.split(),
        capture_output=capture,
        text=True,
    )
    if capture:
        return result.stdout.strip()
    return ""

print(f"Target namespace: {NAMESPACE}")

---
## Step 1 — Enable the MLflow Operator

The MLflow Operator is managed by the RHOAI `DataScienceCluster` CR. If it is already set to `Managed`, skip this step.

In [ ]:
# Check current state of the mlflowoperator component
!oc get datasciencecluster default-dsc \
    -o jsonpath='{.spec.components.mlflowoperator.managementState}'

In [ ]:
# Enable the MLflow Operator if not already Managed
# Run this cell only if the output above was NOT 'Managed'

ENABLE_OPERATOR_PATCH = '''
oc patch datasciencecluster default-dsc \
  --type=merge \
  -p '{"spec":{"components":{"mlflowoperator":{"managementState":"Managed"}}}}'
'''

print("Run this command to enable the MLflow Operator:")
print(ENABLE_OPERATOR_PATCH)
print()
print("Or execute directly:")
!oc patch datasciencecluster default-dsc \
    --type=merge \
    -p '{"spec":{"components":{"mlflowoperator":{"managementState":"Managed"}}}}'

In [ ]:
import time

print("Waiting for MLflow Operator to become ready...")
for attempt in range(12):  # up to ~2 minutes
    result = subprocess.run(
        ["oc", "get", "deployment", "mlflow-operator",
         "-n", "redhat-ods-applications",
         "-o", "jsonpath={.status.availableReplicas}"],
        capture_output=True, text=True
    )
    if result.stdout.strip() == "1":
        print("MLflow Operator is running.")
        break
    print(f"  attempt {attempt+1}/12 — not ready yet, waiting 10s...")
    time.sleep(10)
else:
    print("Operator did not become ready in time. Check: oc get pods -n redhat-ods-applications | grep mlflow")

---
## Step 2 — Create the Project (Namespace)

Each OpenShift project maps 1:1 to an MLflow workspace. Create the project if it does not already exist.

In [ ]:
result = subprocess.run(
    ["oc", "get", "project", NAMESPACE],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"Project '{NAMESPACE}' already exists.")
else:
    subprocess.run(["oc", "new-project", NAMESPACE], check=True)
    print(f"Project '{NAMESPACE}' created.")

---
## Step 3 — Deploy the Minimal MLflowTrackingServer

The Minimal deployment uses:
- **SQLite** as the tracking database (stored on the PVC)
- A **PersistentVolumeClaim** for both the database file and artifact storage
- **1 replica** (no HA)

This is the fastest path to a running server — no external PostgreSQL or S3 bucket required.

### What the CR does

```
MLflowTrackingServer CR
        │
        ▼  (MLflow Operator reconciles)
┌───────────────────────────────────────────┐
│  Deployment: mlflow                        │
│  ┌─────────────────────────────────────┐  │
│  │  Pod: mlflow-<hash>                 │  │
│  │                                     │  │
│  │  Container: mlflow-server           │  │
│  │    - mlflow server (3.13.0)         │  │
│  │    - backend-store: sqlite:///data  │  │
│  │    - artifact-root: /data/artifacts │  │
│  │                      │              │  │
│  │                      ▼              │  │
│  │             PVC: mlflow-data (10Gi) │  │
│  └─────────────────────────────────────┘  │
└───────────────────────────────────────────┘
        │
        ▼  (also creates)
  Service + Route (HTTPS via OpenShift edge TLS)
  RoleBinding (RBAC for workbench service accounts)
```

In [ ]:
MINIMAL_CR = f"""apiVersion: mlflow.opendatahub.io/v1alpha1
kind: MLflowTrackingServer
metadata:
  name: mlflow
  namespace: {NAMESPACE}
  labels:
    app.kubernetes.io/part-of: mlflow-operator
spec:
  # --- Minimal mode: SQLite database + PVC artifact storage ---
  # No external PostgreSQL or S3 bucket required.
  # Suitable for: development, testing, demos, single-user workbenches.
  deploymentMode: minimal

  storage:
    type: pvc
    pvc:
      size: 10Gi            # Adjust as needed (holds DB + artifacts)
      # storageClassName: ""  # Omit to use the cluster default StorageClass

  # Single replica — not HA, but sufficient for dev/test
  replicas: 1

  # Resource limits for the tracking server pod
  resources:
    requests:
      cpu: 200m
      memory: 512Mi
    limits:
      cpu: 1000m
      memory: 2Gi
"""

print("Minimal MLflowTrackingServer CR:")
print(MINIMAL_CR)

In [ ]:
import tempfile, os

# Write CR to a temp file and apply it
with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    f.write(MINIMAL_CR)
    cr_path = f.name

result = subprocess.run(["oc", "apply", "-f", cr_path], capture_output=True, text=True)
os.unlink(cr_path)

if result.returncode == 0:
    print(result.stdout.strip())
else:
    print("ERROR:", result.stderr.strip())
    print()
    print("Common causes:")
    print("  - MLflow Operator not yet running (re-run Step 1)")
    print("  - CRD not yet installed (oc get crd mlflowtrackingservers.mlflow.opendatahub.io)")
    print("  - Insufficient permissions (need edit or admin role in namespace)")

---
## Step 4 — Wait for the Server to Become Ready

In [ ]:
print(f"Waiting for MLflowTrackingServer in namespace '{NAMESPACE}'...")

for attempt in range(24):  # up to ~4 minutes
    # Check CR status
    status_result = subprocess.run(
        ["oc", "get", "mlflowtrackingserver", "mlflow",
         "-n", NAMESPACE,
         "-o", "jsonpath={.status.conditions[?(@.type==\"Ready\")].status}"],
        capture_output=True, text=True
    )
    ready_status = status_result.stdout.strip()

    pod_result = subprocess.run(
        ["oc", "get", "pods", "-n", NAMESPACE, "-l", "app=mlflow",
         "-o", "jsonpath={.items[0].status.phase}"],
        capture_output=True, text=True
    )
    pod_phase = pod_result.stdout.strip()

    print(f"  [{attempt+1:2d}/24] CR Ready={ready_status or 'unknown':8s}  Pod phase={pod_phase or 'pending'}")

    if ready_status == "True" or pod_phase == "Running":
        print("\nMLflow Tracking Server is running!")
        break
    time.sleep(10)
else:
    print("\nServer did not become ready in 4 minutes.")
    print("Investigate with:")
    print(f"  oc describe mlflowtrackingserver mlflow -n {NAMESPACE}")
    print(f"  oc get events -n {NAMESPACE} --sort-by=.lastTimestamp | tail -20")

In [ ]:
# Show what the Operator created
print("=== MLflowTrackingServer CR status ===")
!oc get mlflowtrackingserver mlflow -n {NAMESPACE} -o wide

print()
print("=== Pods ===")
!oc get pods -n {NAMESPACE} -l app=mlflow

print()
print("=== PersistentVolumeClaims ===")
!oc get pvc -n {NAMESPACE}

print()
print("=== Service ===")
!oc get svc -n {NAMESPACE} -l app=mlflow

print()
print("=== Route (HTTPS tracking URL) ===")
!oc get route -n {NAMESPACE} -l app=mlflow

---
## Step 5 — Extract the Tracking URL

In [ ]:
route_host = subprocess.run(
    ["oc", "get", "route", "-n", NAMESPACE, "-l", "app=mlflow",
     "-o", "jsonpath={.items[0].spec.host}"],
    capture_output=True, text=True
).stdout.strip()

if route_host:
    TRACKING_URI = f"https://{route_host}/workspaces/{NAMESPACE}"
    print(f"MLflow Tracking URI : {TRACKING_URI}")
    print()
    print("Set this in notebook 01:")
    print(f'  MLFLOW_TRACKING_URI = "{TRACKING_URI}"')
    print()
    print("Or export for use outside the workbench:")
    print(f'  export MLFLOW_TRACKING_URI="{TRACKING_URI}"')
    print(f'  export MLFLOW_TRACKING_AUTH="kubernetes-namespaced"')
else:
    print("Could not retrieve route — check 'oc get route -n", NAMESPACE, "'")

---
## Step 6 — Grant Access (RBAC)

RHOAI 3.5 uses three aggregate ClusterRoles for MLflow access:

| ClusterRole | Grants |
|-------------|--------|
| `mlflow-view` | Read-only access to experiments, runs, models |
| `mlflow-edit` | Create/update experiments, runs, log metrics/params/artifacts |
| `mlflow-integration` | Additional access needed by workbench service accounts |

The workbench annotation (`opendatahub.io/mlflow-instance`) automatically creates the `mlflow-integration` RoleBinding.
Grant `mlflow-edit` to human users who need to track experiments.

In [ ]:
# Replace <username> with the actual OpenShift username or service account
GRANT_USER = "<username>"   # e.g. "developer" or "system:serviceaccount:my-ns:my-sa"

RBAC_COMMANDS = f"""
# Grant edit (track experiments) to a user:
oc adm policy add-role-to-user mlflow-edit {GRANT_USER} -n {NAMESPACE}

# Grant view-only access:
oc adm policy add-role-to-user mlflow-view {GRANT_USER} -n {NAMESPACE}

# Check current RoleBindings for MLflow:
oc get rolebindings -n {NAMESPACE} | grep mlflow
"""

print("Run these commands to grant access:")
print(RBAC_COMMANDS)

---
## Step 7 — Smoke Test: List Workspaces

In [ ]:
import os
import mlflow

if route_host:
    os.environ["MLFLOW_TRACKING_URI"]  = TRACKING_URI
    os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes-namespaced"

    mlflow.set_tracking_uri(TRACKING_URI)

    try:
        workspaces = mlflow.list_workspaces()
        print(f"Workspace accessible: {[w.name for w in workspaces]}")
        print()
        print("Minimal deployment is ready. Proceed to notebook 01_setup_authentication.ipynb.")
    except Exception as e:
        print(f"Smoke test failed: {e}")
        print()
        print("Ensure you have mlflow-view or mlflow-edit RoleBinding in this namespace.")
else:
    print("Skipping smoke test — route not available yet.")

---
## Reference: Production Configuration

When you are ready to move beyond the Minimal deployment, use the Production configuration.
This requires an external PostgreSQL instance and an S3-compatible bucket to exist beforehand.

Key differences from Minimal:
- `replicas: 2` — survives pod restarts without data loss
- `serveArtifacts: true` — MLflow proxies artifact uploads/downloads through itself (required when workbench pods cannot reach S3 directly)
- External PostgreSQL — fully managed, backed up, HA
- S3 artifact store — durable, unlimited storage, compatible with Trace Archival

In [ ]:
PRODUCTION_CR = f"""apiVersion: mlflow.opendatahub.io/v1alpha1
kind: MLflowTrackingServer
metadata:
  name: mlflow
  namespace: {NAMESPACE}
spec:
  # --- Production mode: PostgreSQL + S3 ---
  deploymentMode: production

  # Two replicas for availability
  replicas: 2

  # Proxy artifact uploads/downloads through MLflow (required if pods can't reach S3 directly)
  serveArtifacts: true

  database:
    type: postgresql
    postgresql:
      host: postgresql.{NAMESPACE}.svc.cluster.local
      port: 5432
      databaseName: mlflow
      credentialsSecret:
        name: mlflow-db-secret    # Must exist; keys: user, password

  storage:
    type: s3
    s3:
      bucket: my-mlflow-artifacts
      # prefix: mlflow              # optional key prefix
      credentialsSecret:
        name: mlflow-s3-secret     # Must exist; keys: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY
      # endpointUrl: https://minio.example.com  # only needed for non-AWS S3

  # RHOAI 3.5: Trace Archival — moves old span payloads from PostgreSQL to S3
  traceArchival:
    enabled: true
    schedule: "0 2 * * *"         # Run at 2 AM daily
    retention: 30d                 # Archive spans older than 30 days
    maxTracesPerPass: 1000         # Batch size per CronJob run
    location:
      bucket: my-mlflow-artifacts
      prefix: trace-archive

  resources:
    requests:
      cpu: 500m
      memory: 1Gi
    limits:
      cpu: 2000m
      memory: 4Gi
"""

print("Production MLflowTrackingServer CR (for reference — do not apply without the prerequisite secrets):")
print(PRODUCTION_CR)

---
## Reference: Per-Project S3 Override (MLflowConfig)

If your cluster has a shared MLflow server but individual projects need their own S3 bucket for artifacts, use the `MLflowConfig` CR (namespace-scoped). This overrides the server-level artifact storage for one project without touching the `MLflowTrackingServer` CR.

In [ ]:
MLFLOWCONFIG_CR = f"""# Step A: Secret with S3 credentials for this project
apiVersion: v1
kind: Secret
metadata:
  name: mlflow-artifact-connection    # Name must be exactly 'mlflow-artifact-connection'
  namespace: {NAMESPACE}
type: Opaque
stringData:
  AWS_ACCESS_KEY_ID: <your-access-key>
  AWS_SECRET_ACCESS_KEY: <your-secret-key>
  AWS_DEFAULT_REGION: us-east-1
  # AWS_ENDPOINT_URL: https://minio.example.com  # for non-AWS S3
---
# Step B: MLflowConfig referencing the secret
apiVersion: mlflow.opendatahub.io/v1alpha1
kind: MLflowConfig
metadata:
  name: mlflow                        # Name must be exactly 'mlflow'
  namespace: {NAMESPACE}
spec:
  artifactStorage:
    s3:
      bucket: project-specific-bucket
      prefix: {NAMESPACE}             # isolate artifacts by namespace
      credentialsSecret:
        name: mlflow-artifact-connection
"""

print("MLflowConfig CR for per-project S3 artifact storage override:")
print(MLFLOWCONFIG_CR)

print()
print("Apply with: oc apply -f mlflowconfig.yaml")
print()
print("NOTE: The secret and config names are fixed by the Operator.")
print("      The override takes effect on the next server restart or within ~60 seconds.")

---
## Troubleshooting the Minimal Deployment

| Symptom | Command to investigate | Common fix |
|---------|------------------------|------------|
| Pod stuck in `Pending` | `oc describe pod -n <ns> -l app=mlflow` | No PVC provisioner; set `storageClassName` explicitly |
| Pod stuck in `CrashLoopBackOff` | `oc logs -n <ns> -l app=mlflow --previous` | SQLite file permissions; check PVC mount |
| Route returns 503 | `oc get endpoints -n <ns>` | Pod not ready; check liveness probe |
| CR stays in `Progressing` | `oc describe mlflowtrackingserver mlflow -n <ns>` | Operator reconcile error; check operator logs |
| `403` from SDK | `oc get rolebindings -n <ns> \| grep mlflow` | Missing RBAC; run Step 6 |

Useful diagnostic commands:
```bash
# Full CR status
oc get mlflowtrackingserver mlflow -n <ns> -o yaml

# Operator logs
oc logs -n redhat-ods-applications deployment/mlflow-operator --tail=50

# Server logs
oc logs -n <ns> deployment/mlflow --tail=50

# All events in namespace (sorted newest last)
oc get events -n <ns> --sort-by=.lastTimestamp
```

---
## Summary

| Step | What was done |
|------|---------------|
| 1 — Enable Operator | `DataScienceCluster` patched: `mlflowoperator: Managed` |
| 2 — Create project | `oc new-project` — 1:1 namespace = workspace |
| 3 — Deploy Minimal CR | SQLite + PVC, 1 replica, no external dependencies |
| 4 — Wait for readiness | Operator creates Deployment, Service, Route, PVC |
| 5 — Extract URL | `https://<route>/workspaces/<namespace>` |
| 6 — Grant RBAC | `mlflow-edit` or `mlflow-view` RoleBinding for users |
| 7 — Smoke test | `mlflow.list_workspaces()` confirms connectivity |

**Next:** Run `01_setup_authentication.ipynb` using the `TRACKING_URI` extracted in Step 5.

---
## Custom Workbench Image (Disconnected Clusters)

Standard RHOAI workbench images do not include the `mlflow[kubernetes]` SDK. On a disconnected cluster there is no egress to install it at runtime, so the SDK must be baked into the workbench image.

### Build the image (on a connected machine)

Create a `Containerfile`:

```dockerfile
# Match the RHOAI 3.5 base image used in your cluster.
# Common choices (check your cluster mirror for the exact digest):
#   registry.redhat.io/rhoai/odh-generic-data-science-notebook-rhel9:v3.5
#   registry.redhat.io/rhoai/odh-pytorch-notebook-rhel9:v3.5
FROM registry.redhat.io/rhoai/odh-generic-data-science-notebook-rhel9:v3.5

USER 0
RUN pip install --no-cache-dir "mlflow[kubernetes]>=3.11" "packaging>=23"
USER 1001
```

Build and push:

```bash
# Build on a connected machine
podman build -t <internal-registry>/<namespace>/mlflow-workbench:3.5 .

# Push to your cluster's internal registry (or Quay mirror)
podman push <internal-registry>/<namespace>/mlflow-workbench:3.5
```

For a fully air-gapped build (no pip access even during build), download wheels first:

```bash
# On a connected machine — collect all wheels
pip download "mlflow[kubernetes]>=3.11" "packaging>=23" \
    --platform manylinux_2_28_x86_64 \
    --python-version 311 \
    --only-binary=:all: \
    -d ./wheels/

# Add to Containerfile instead of the pip install line above:
#   COPY wheels/ /tmp/wheels/
#   RUN pip install --no-index --find-links=/tmp/wheels \
#         "mlflow[kubernetes]>=3.11" "packaging>=23"
```

### Register the image in RHOAI

```bash
# Label the ImageStream so RHOAI shows it in the workbench image list
oc import-image mlflow-workbench:3.5 \
  --from=<internal-registry>/<namespace>/mlflow-workbench:3.5 \
  --confirm \
  -n redhat-ods-applications

oc label imagestream mlflow-workbench \
  opendatahub.io/notebook-image=true \
  -n redhat-ods-applications
```

Or via the RHOAI Dashboard: **Settings > Notebook images > Add notebook image** — paste the image pull spec and give it a display name like `MLflow 3.13 (DS Python 3.11)`.

Once the image is registered, select it when creating a new workbench in your project.